#Basic Tasks

In [0]:
%sql
create catalog if not exists cyntexa_dev;

create schema if not exists cyntexa_dev.sales;

In [0]:
%sql
create or replace table cyntexa_dev.sales.orders_raw(
    id int,
    customer_id string,
    email string,
    price double,
    status string,
    date date
)

In [0]:
%sql
INSERT INTO cyntexa_dev.sales.orders_raw (id, customer_id, email, price, status, date) VALUES
(1, "CUST101", 'john.doe@gmail.com', 249.99, 'completed', '2026-08-01'),
(2, "CUST102", 'sarah.smith@gmail.com', 89.50, 'completed', '2026-08-02'),
(3, "CUST103", 'mike.jones@yahoo.com', 1299.00, 'pending', '2026-08-03'),
(4, "CUST104", 'priya.singh@gmail.com', 45.75, 'cancelled', '2026-08-04'),
(5, "CUST105", 'alex.brown@outlook.com', 599.99, 'completed', '2026-08-05'),
(6, "CUST101", 'john.doe@gmail.com', 129.00, 'completed', '2026-08-06'),
(7, "CUST106", 'emma.wilson@gmail.com', 75.25, 'pending', '2026-08-07'),
(8, "CUST107", 'raj.patel@yahoo.com', 899.99, 'completed', '2026-08-08'),
(9, "CUST108", 'lisa.chen@gmail.com', 34.99, 'cancelled', '2026-08-09'),
(10, "CUST102", 'sarah.smith@gmail.com', 199.50, 'completed', '2026-08-10');

In [0]:
%sql
create or replace view cyntexa_dev.sales.orders_view as
select * from cyntexa_dev.sales.orders_raw where status = "completed"

In [0]:
%sql
select * from cyntexa_dev.sales.orders_view

In [0]:
%sql
select * from samples.tpch.orders limit 10

In [0]:
%sql
select * from samples.tpch.orders where o_orderstatus = "O" limit 5

In [0]:
%sql
select * from samples.tpch.orders where o_custkey is null

#Intermediate Tasks

In [0]:
%sql
create or replace function cyntexa_dev.sales.masked_values(inputStr STRING)
returns string
return case
    when inputStr is null then null
    when length(inputStr) <= 4 then repeat('*', length(inputStr))
    when inputStr like '%@%' then
        concat(
            substring(split(inputStr, '@')[0], 1, length(split(inputStr, '@')[0]) - 4),
            '****',
            '@',
            split(inputStr, '@')[1]
        )
    else concat(substring(inputStr, 1, length(inputStr) - 4), '****')
end

In [0]:
%sql
select id, cyntexa_dev.sales.masked_values(customer_id) as masked_customer_id, 
cyntexa_dev.sales.masked_values(email) as masked_email, price, status, date 
from cyntexa_dev.sales.orders_raw limit 5

In [0]:
%sql
create or replace view cyntexa_dev.sales.customers as
select distinct
    customer_id,
    email
from cyntexa_dev.sales.orders_raw;

In [0]:
%sql
create or replace view cyntexa_dev.sales.total_spend_per_customer as
select c.customer_id, c.email, sum(o.price) as total_spend
from cyntexa_dev.sales.orders_raw o join cyntexa_dev.sales.customers c
on o.customer_id = c.customer_id
group by c.customer_id, c.email
    


In [0]:
%sql
select * from cyntexa_dev.sales.total_spend_per_customer

#Advanced Tasks

In [0]:
%sql
create catalog if not exists cyntexa_staging;
create catalog if not exists cyntexa_prod;

create schema if not exists cyntexa_staging.sales;
create schema if not exists cyntexa_prod.sales;

-- the above catalog will hold the schemas its like catalog is a container which contains the schemas and later this schemas will going to hold the tables, views, functions etc

8. 
In real production there are mainly 3 catalogs - dev,uat/staging and prod

dev - this catalog is used in deveopment phase.developers used to write code , add new features, delete old features.

uat/statging - 
This catalog used for testing environment .In this the features or new updates are tested with different test cases.if the feature pass all test cases it sends to the production else developers fix the bug or optimize it.

prod - 
in prod environment , the application or feature is open for the use for users

In [0]:
%sql
WITH customer_revenue AS (
    SELECT 
        r.r_name AS region_name,
        c.c_custkey AS customer_id,
        c.c_name AS customer_name,
        SUM(o.o_totalprice) AS total_revenue,
        DENSE_RANK() OVER (
            PARTITION BY r.r_name 
            ORDER BY SUM(o.o_totalprice) DESC
        ) AS revenue_rank
    FROM samples.tpch.customer c
    JOIN samples.tpch.orders o ON c.c_custkey = o.o_custkey
    JOIN samples.tpch.nation n ON c.c_nationkey = n.n_nationkey
    JOIN samples.tpch.region r ON n.n_regionkey = r.r_regionkey
    GROUP BY 
        r.r_name, 
        c.c_custkey, 
        c.c_name
)
SELECT 
    region_name,
    customer_id,
    customer_name,
    total_revenue,
    revenue_rank
FROM customer_revenue
WHERE revenue_rank <= 5
ORDER BY 
    region_name, 
    revenue_rank;